In [6]:
import csv
import os
import random
from datetime import datetime, timedelta
from typing import List, Tuple, Union


def generate_sales_csv(
    output_path: str = "data/sales.csv",
    num_rows: int = 50,
    start_date: datetime = datetime(2023, 11, 1),
    debug: bool = True
) -> str:
    """Generate a synthetic sales dataset and write it to a CSV file.

    This function creates randomized sales records for testing or demo purposes,
    including a few intentionally malformed rows to test data validation workflows.

    Args:
        output_path (str): Path where the CSV file will be written.
        num_rows (int): Number of valid rows to generate before adding malformed ones.
        start_date (datetime): Starting date for randomly generated sales dates.
        debug (bool, optional): If True, prints detailed debug information. Defaults to True.

    Returns:
        str: The absolute path to the generated CSV file.

    Raises:
        OSError: If there is an issue creating directories or writing the file.
    """
    start_time = datetime.now()
    if debug:
        print(f"[{start_time.strftime('%Y-%m-%d %H:%M:%S')}] 🔧 Starting CSV generation...")

    # === Configuration ===
    product_codes: List[str] = ["PRD-A12", "PRD-B07", "PRD-C03", "PRD-D55"]
    statuses: List[str] = ["Completed", "Canceled"]

    # === Step 1: Ensure output directory exists ===
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        if debug:
            print(f"📂 Directory ensured: {os.path.dirname(output_path)}")
    except OSError as e:
        raise OSError(f"Failed to create output directory: {e}")

    # === Step 2: Generate valid rows ===
    rows: List[List[Union[str, float]]] = []
    if debug:
        print("🧪 Generating valid sales rows...")

    for i in range(num_rows):
        if debug and i % 10 == 0:
            print(f"  ...generating row {i}/{num_rows}")

        order_number = f"ORD-{1001 + i}"
        customer_id = f"CUST-{200 + random.randint(1, 20)}"
        date = start_date + timedelta(days=random.randint(0, 59))
        product_code = random.choice(product_codes)
        amount = round(random.uniform(20.0, 500.0), 2)
        status = random.choices(statuses, weights=[0.85, 0.15])[0]

        rows.append([
            order_number,
            customer_id,
            amount,
            date.strftime("%Y-%m-%d"),
            product_code,
            status,
        ])

    # === Step 3: Add malformed rows ===
    if debug:
        print("➕ Adding malformed rows...")
    malformed_rows: List[List[Union[str, float]]] = [
        ["ORD-9999", "CUST-999", "not_a_number", "2023-12-10", "PRD-A12", "Completed"],
        ["ORD-1000", "CUST-998", 400.00, "bad_date", "PRD-B07", "Completed"],
        ["ORD-1001", "CUST-997", 275.00, "2023-12-12", "", "Completed"],
    ]
    rows.extend(malformed_rows)

    # === Step 4: Write rows to CSV ===
    try:
        with open(output_path, mode="w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(
                ["order_number", "customer_id", "transaction_amount", "date", "product_code", "status"]
            )
            writer.writerows(rows)

        if debug:
            print(f"💾 Wrote {len(rows)} records (including malformed) to {output_path}")

    except OSError as e:
        raise OSError(f"Error writing to CSV file: {e}")

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    print(f"✅ CSV generation complete. ({duration:.2f}s) → {output_path}")

    return os.path.abspath(output_path)


control prompt: 

Hey — can you help me summarize some sales data? 

We’ve got a CSV file with order-level transactions. I need you to generate python code for me to run locally to get at the following insights: 

1. Total sales grouped by product_code 
2. Total sales grouped by customer_id 
3. Monthly sales totals based on the date column 

You can skip rows where status isn’t "Completed" — we’re only looking at finalized transactions. 

The file is located at C:/data/sales.csv 

and includes these columns: 
- order_number 
- customer_id 
- transaction_amount 
- date 
- product_code 
- status 

Just output everything to the terminal in a clean format — I’ll copy/paste it and run it locally.

In [7]:
# ChatGPT - Free 1/10/2026

import pandas as pd

# Load the CSV
df = pd.read_csv("./data/sales.csv")

# Ensure date column is parsed as datetime
df["date"] = pd.to_datetime(df["date"])

# Filter to completed transactions only
df_completed = df[df["status"] == "Completed"]

# 1. Total sales by product_code
sales_by_product = (
    df_completed
    .groupby("product_code", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("transaction_amount", ascending=False)
)

# 2. Total sales by customer_id
sales_by_customer = (
    df_completed
    .groupby("customer_id", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("transaction_amount", ascending=False)
)

# 3. Monthly sales totals
df_completed["month"] = df_completed["date"].dt.to_period("M")
monthly_sales = (
    df_completed
    .groupby("month", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("month")
)

# Output results
print("\n=== Total Sales by Product Code ===")
print(sales_by_product.to_string(index=False))

print("\n=== Total Sales by Customer ID ===")
print(sales_by_customer.to_string(index=False))

print("\n=== Monthly Sales Totals ===")
print(monthly_sales.to_string(index=False))



=== Total Sales by Product Code ===
product_code  transaction_amount
     PRD-C03             4988.34
     PRD-A12             3646.37
     PRD-B07             3089.68
     PRD-D55             2154.54

=== Total Sales by Customer ID ===
customer_id  transaction_amount
   CUST-207             1579.86
   CUST-218             1403.14
   CUST-211             1398.43
   CUST-220             1309.75
   CUST-210             1186.32
   CUST-215              940.89
   CUST-204              869.30
   CUST-201              863.65
   CUST-209              852.86
   CUST-213              823.42
   CUST-216              773.33
   CUST-217              528.30
   CUST-202              463.99
   CUST-203              430.94
   CUST-208              395.47
   CUST-214               38.75
   CUST-205               20.53

=== Monthly Sales Totals ===
  month  transaction_amount
2023-11             4913.47
2023-12             8965.46


In [8]:
import pandas as pd
import time

def summarize_sales_data(file_path="./data/sales_with_errors.csv", debug=True):
    """
    Summarize sales data from a CSV file by product, customer, and month.

    This function loads order-level transaction data, filters for completed transactions,
    and generates the following summaries:
      1. Total sales grouped by product_code
      2. Total sales grouped by customer_id
      3. Monthly total sales based on the transaction date

    Parameters
    ----------
    file_path : str, optional
        Path to the CSV file containing sales data.
    debug : bool, optional
        If True, prints detailed logging of data loading, filtering, coercion, and aggregation steps.

    Returns
    -------
    None
        Prints summary tables to the console.

    Raises
    ------
    FileNotFoundError
        If the specified file_path cannot be found.
    ValueError
        If required columns are missing from the CSV.
    
    Example
    -------
    >>> summarize_sales_data("C:/data/sales_with_errors.csv", debug=True)
    """

    start_time = time.time()
    if debug:
        print(f"\n[INFO] Starting sales summary at {time.ctime(start_time)}")
        print(f"[INFO] Reading CSV file: {file_path}")

    # Load data
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"File not found: {file_path}") from e

    # Validate required columns
    required_cols = ["order_number", "customer_id", "transaction_amount", "date", "product_code", "status"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Filter completed transactions
    df = df[df["status"].str.lower() == "completed"]
    if debug:
        print(f"[DEBUG] Filtered to {len(df)} completed transactions.")

    # Coerce numeric values for transaction_amount
    df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
    dropped_rows = df["transaction_amount"].isna().sum()
    df = df.dropna(subset=["transaction_amount"])
    if debug:
        if dropped_rows > 0:
            print(f"[DEBUG] Dropped {dropped_rows} rows with non-numeric transaction_amount values.")
        print("[DEBUG] Numeric coercion complete — calculations successful.")

    # Convert date to datetime and extract month
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["month"] = df["date"].dt.to_period("M")

    # --- 1. Total sales by product_code ---
    sales_by_product = df.groupby("product_code")["transaction_amount"].sum().reset_index()
    sales_by_product = sales_by_product.sort_values(by="transaction_amount", ascending=False)

    # --- 2. Total sales by customer_id ---
    sales_by_customer = df.groupby("customer_id")["transaction_amount"].sum().reset_index()
    sales_by_customer = sales_by_customer.sort_values(by="transaction_amount", ascending=False)

    # --- 3. Monthly total sales ---
    monthly_sales = df.groupby("month")["transaction_amount"].sum().reset_index()
    monthly_sales = monthly_sales.sort_values(by="month")

    # Print results to console
    print("\n===== TOTAL SALES BY PRODUCT =====")
    print(sales_by_product.to_string(index=False))
    print("\n===== TOTAL SALES BY CUSTOMER =====")
    print(sales_by_customer.to_string(index=False))
    print("\n===== MONTHLY SALES TOTALS =====")
    print(monthly_sales.to_string(index=False))

    # Runtime summary
    end_time = time.time()
    duration = round(end_time - start_time, 2)
    print(f"\n[INFO] Sales summary complete at {time.ctime(end_time)}")
    print(f"[INFO] Runtime: {duration} seconds")


if __name__ == "__main__":
    summarize_sales_data(debug=True)



[INFO] Starting sales summary at Sat Jan 10 14:55:23 2026
[INFO] Reading CSV file: ./data/sales_with_errors.csv
[DEBUG] Filtered to 49 completed transactions.
[DEBUG] Dropped 1 rows with non-numeric transaction_amount values.
[DEBUG] Numeric coercion complete — calculations successful.

===== TOTAL SALES BY PRODUCT =====
product_code  transaction_amount
     PRD-C03             4352.04
     PRD-A12             3447.51
     PRD-B07             3094.22
     PRD-D55             1638.22

===== TOTAL SALES BY CUSTOMER =====
customer_id  transaction_amount
   CUST-206             1361.28
   CUST-204             1274.01
   CUST-216             1165.95
   CUST-211             1158.25
   CUST-207             1035.00
   CUST-205              894.21
   CUST-214              748.68
   CUST-208              633.24
   CUST-203              612.92
   CUST-202              525.31
   CUST-209              472.75
   CUST-218              388.81
   CUST-215              357.47
   CUST-219              3